# Main overview

You potentially need to `pip install pandas` first.

Just run the cell below. Click on the output, not on the code part of the cell; that opens the whole code. If you accidentally open it, go `View -> Collapse Selected Code` to close it again.

In [86]:
group = 'rigi'

import re
from getpass import getuser
from collections import Counter, defaultdict
from itertools import chain, repeat
from pathlib import Path
from multiprocessing import Pool

import pandas as pd
import rich
from rich.table import Table
from rich.text import Text


# Workaround to fix weird scroll-inducing whitespace at end of many cell's outputs.
from IPython.display import display_html, display
display_html("""<style>.jp-OutputArea { max-height: none !important; overflow-y: visible !important }</style>""", raw=True)


########################
# READ ALL ACTIVE JOBS #


# We need to go with length and cut by lenght, because some entries may have spaces, some may be too long, etc.
jobs = !squeue -A {group} -O JobId:20,Name:20,UserName:20,State:20,TimeUsed:20,NumCPUs:20,QOS:20,NumNodes:20,GRES:20,RestartCnt:20,Reason:20
jobs = [[j[i*20:(i+1)*20].strip() for i in range(11)] for j in jobs]
jobs = pd.DataFrame(jobs[1:], columns=jobs[0])



##########################
# READ ALL WORKDIRS EVER #

# _xid_re = re.compile(r'(\d\d)(\d\d)_(\d\d)(\d\d)(\d\d)')
_xid_re = re.compile(r'\d\d\d\d_\d\d\d\d\d\d')
def extract_xid(name):
    if firstmatch := _xid_re.search(name):
        return firstmatch.group()
    return None


basedir = Path('/checkpoint/rigi/bv2/workdirs')
workdirs = [d.name for d in basedir.iterdir() if d.is_dir()]
wd_by_xid = {xid: wd for wd in workdirs if (xid := extract_xid(wd))}

#####################################
# SPLIT INTO CURRENT / RECENT / OLD #
# We do this split to add much more info to recent, and less to old.

NUM_RECENT = 50

hot_xids = {}
cold_xids = {}
for xid, wd in sorted(wd_by_xid.items(), reverse=True):
    states = Counter(jobs[jobs.NAME == xid].STATE)
    if states:
        hot_xids[xid] = {"states": states, "wd": wd}
    else:
        cold_xids[xid] = wd

frozen_xids = {xid: {"wd": cold_xids[xid]} for xid in list(cold_xids)[NUM_RECENT:]}
cold_xids = {xid: {"wd": cold_xids[xid]} for xid in list(cold_xids)[:NUM_RECENT]}

def _extra_info(xid, info):
    info["user"] = (basedir / info["wd"]).owner()
    launchinfo = basedir / info["wd"] / 'launchinfo.txt'
    if launchinfo.is_file():  # Launched with our sweep launcher
        info["wus"] = {}
        for wuwd in (basedir / info["wd"]).iterdir():
            if wuwd.is_dir():
                info["wus"][wuwd] = (wuwd / "DONE").exists()
        info["config"] = next(re.finditer(r"bv2/config/(.*?) ", launchinfo.read_text())).group(1)
    return xid, info

with Pool() as pool:
    hot_xids = dict(pool.starmap(_extra_info, hot_xids.items()))
    cold_xids = dict(pool.starmap(_extra_info, cold_xids.items()))


#########################
# PREPARE VISUALIZATION #

STATE_NAMES = {
    "RUNNING": Text("Run", "blue"),
    "PENDING": Text("Pend", "yellow"),
    "REQUEUE_HOLD": Text("Hold", "red"),
    "COMPLETING": Text("End", "green"),
    # And our own for old jobs:
    True: Text("Done", "green"),
    False: Text("Fail", "red"),
}
# STATE_NAMES = {"RUNNING": "🏃", "PENDING": "⏳", "REQUEUE_HOLD": "🚫"}  # Sadly misaligns columns.

nGPUs = {f'gres/gpu:{i}': i for i in range(1, 9)}

tblH = Table(show_header=True, header_style="bold magenta", show_footer=True, footer_style="bold magenta", box=rich.box.HORIZONTALS, collapse_padding=True)
tblH.add_column("xid", justify="left")
tblH.add_column("usr", justify="left")
tblH.add_column("states", justify="left")
tblH.add_column("wus", justify="right")
# tblH.add_column("N", justify="right")
tblH.add_column("gpu", justify="right")
tblH.add_column("gpus", justify="right")
tblH.add_column("r", justify="right")
tblH.add_column("QoS", justify="left")
tblH.add_column("config", justify="left")

all_states = Counter()
all_total_wus = 0
all_total_gpus = 0
for xid, info in hot_xids.items():
    xjobs = jobs[jobs.NAME == xid]
    qos = ' '.join(xjobs.QOS.unique().tolist())
    users = ' '.join(xjobs.USER.unique().tolist())
    nodes = ' '.join(xjobs.NODES.unique().tolist())
    gpus = int(nodes) * nGPUs.get(xjobs.TRES_PER_NODE.unique().tolist()[0], 0)
    total_gpus = gpus * info["states"]["RUNNING"]
    max_restarts = str(xjobs.RESTART_COUNT.max())

    all_total_wus += len(info["wus"])
    all_total_gpus += total_gpus

    ndone = sum(info['wus'].values())
    states = info["states"] + Counter({True: ndone, False: len(info['wus']) - ndone - sum(info["states"].values())})
    all_states.update(states)
    states = Text(' ').join(STATE_NAMES[s] + Text(f":{n}") for s, n in states.most_common())
    tblH.add_row(xid, users, states, str(len(info["wus"])), str(gpus), str(total_gpus), max_restarts, qos, info["config"])

tblH.columns[2].footer = Text(' ').join(STATE_NAMES[s] + Text(f":{n}") for s, n in all_states.most_common())
tblH.columns[3].footer = str(all_total_wus)
tblH.columns[5].footer = str(all_total_gpus)

tblC = Table(show_header=True, header_style="bold magenta", box=rich.box.HORIZONTALS)
tblC.add_column("xid", justify="left")
tblC.add_column("usr", justify="left")
tblC.add_column("wus", justify="right")
for xid, info in cold_xids.items():
    if info["user"] != getuser():
        continue
    states = []
    if nfail := sum(1 for v in info["wus"].values() if v is False):
        states.append(Text(f"{nfail}", "red"))
    if ngood := sum(1 for v in info["wus"].values() if v is True):
        states.append(Text(f"{ngood}", "green"))
    tblC.add_row(xid, info["user"], Text("+").join(states))

rich.print('All currently active experiments:', tblH,
           f'YOUR ({getuser()}) experiments in the most recent {NUM_RECENT} inactive ones:', tblC,
           f'Very old experiments: [{len(frozen_xids)} not shown]')

All currently active experiments:
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  xid          usr   states                        wus  gpu  gpus  r  QoS             config                       
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  1211_134919  qkv   Run:48                         48    8   384  0  h200_lowest     finevision.py                
  1211_113900  zhai  Run:1                           1   16    16  1  h200_lowest     finevision_mix.py            
  1211_113850  zhai  Run:1                           1   16    16  1  h200_lowest     finevision.py                
  1211_113501  zhai  Run:1                           1    8     8  1  h200_lowest     finevision_mix.py            
  1211_113444  zhai  Run:1                           1    8     8  1  h200_lowest     finevision.py                
  1211_113151  zhai  Run:1                           1    8     8  0  h100_rigi_high  finevision_mix.py            
  1211_101838  zhai  Run:1                           1   64    64  0  h100_rigi_high  finevision_8n_full.py        
  1209_164235  zhai  Fail:45 Hold:3                 48    8     0  5  h200_lowest     single_task_from_scratch.py  
  1205_144616  zhai  Fail:4 Hold:1                   5   16     0  5  h200_lowest     finevision.py                
  1205_105206  zhai  Run:2                           2   64   128  0  h100_rigi_high  finevision_8n.py             
  1202_122600  qkv   Hold:91 Done:6                 48    8     0  5  h200_lowest     finevision.py                
  1117_163432  qkv   Done:114 Fail:4 Hold:2        120    8     0  5  h100_lowest     code.py                      
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
                     Done:120 Hold:97 Run:56       277        632                                                  
                     Fail:53                                                                                       
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
YOUR (qkv) experiments in the most recent 50 inactive ones:
 ───────────────────────── 
  xid           usr   wus  
 ───────────────────────── 
  1211_134834   qkv        
  1211_110146   qkv     8  
  1210_205120   qkv    48  
  1210_203708   qkv    33  
  1210_203514   qkv        
  1209_091401   qkv     9  
 ───────────────────────── 
Very old experiments: [353 not shown]

# Closer look at single XID

Same story about hiding cell code input as above.

In [88]:
import ipywidgets as widgets
from IPython.display import clear_output, display, display_html
from functools import partial

import json
import shlex
import sws

def load_config(wd, basedir):
    workdir = basedir / wd
    return sws.Config(**json.loads((workdir / "config.json").read_text()), workdir=workdir).finalize()

def last_metric(wd, basedir):
    fname = basedir / wd / "metrics.jsonl"
    last_line = !tail -n 1 {fname}
    try:
        return json.loads(last_line.s)
    except:
        return {}

def load_sacct(jid):
    js = !sacct --long --jobs={jid} --json
    return json.loads('\n'.join(js))['jobs'][0]

def get_xid_info(XID):
    # We need to go with length and cut by lenght, because some entries may have spaces, some may be too long, etc.
    xid_jobs = !squeue -n {XID} -O JobId:20,Name:20,UserName:20,State:20,TimeUsed:20,NumCPUs:20,QOS:20,NumNodes:20,GRES:20,RestartCnt:20,Reason:20
    xid_jobs = [[j[i*20:(i+1)*20].strip() for i in range(11)] for j in xid_jobs]
    xid_jobs = pd.DataFrame(xid_jobs[1:], columns=xid_jobs[0])
    xid_jobs
    
    basedir = Path('/checkpoint/rigi/bv2/workdirs') / XID
    
    launch_command = (basedir / "launchinfo.txt").read_text()
    
    workdirs = [d.name for d in basedir.iterdir() if d.is_dir()]

    with Pool() as pool:
        configs = {c.wid: c for c in pool.map(partial(load_config, basedir=basedir), workdirs)}
        saccts = dict(zip(list(configs), pool.map(load_sacct, [v["jid"] for v in configs.values()])))
        last_metrics = dict(zip(list(configs), pool.map(partial(last_metric, basedir=basedir), workdirs)))

    status = {}
    for wid, c in configs.items():
        if (basedir / c["name"] / "DONE").is_file():
            status[wid] = "DONE"
        elif (jid := c["jid"]) in xid_jobs.JOBID:
            status[wid] = xid_jobs.query(f'JOBID == {jid}').STATE.to_numpy().item()
        elif saccts[wid]['state']['current']:
            # ideally, if it's ERROR, we would try to surface the error from the logs.
            status[wid] = saccts[wid]['state']['current'][-1]
        else:  # Unclear!
            status[wid] = "UNKNOWN"

    return configs, status, saccts, last_metrics

def get_sws_args(submit_line):
    ignore = ["xid:=", "wid:=", "jid:=", "name:="]
    return [a for a in shlex.split(submit_line) if "=" in a and not a.startswith("--") if not any(x in a for x in ignore)]

height = 512

css = r"""<style>
#mytbl {
  border-collapse: collapse;
  width: 100%;
  font-family: system-ui, -apple-system, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
  font-size: 13px;
  table-layout: fixed;   /* stabilizes column widths */
}
#mytbl th, #mytbl td { padding: 4px 8px; }
#mytbl thead, #mytbl tbody tr {
  display: table;
  width: 100%;
  table-layout: fixed;   /* align columns */
}
#mytbl tbody {
  display: block;        /* becomes the scroll area */
  max-height: """ + str(height - 32) + r"""px;
  overflow-y: auto;
}
#mytbl thead th {
  background: #f7f7f9;
  text-align: right;
}
#mytbl tbody tr:nth-child(even) { background: #f5f5f5; }
#mytbl tbody tr:hover { background: #e6f2ff; }

#mytbl .wid { width: 2em; text-align: right; }
#mytbl .jid { width: 4em; text-align: right; }
#mytbl .restarts { width: 0.5em; text-align: center; }
#mytbl .exitcode { width: 0.5em; text-align: center; }
#mytbl .status { width: 0.5em; text-align: center; }
#mytbl .step {width: 3em; text-align: right; }
#mytbl .metric {width: 2em; text-align: right; }
#mytbl .conf { text-align: left; }
#mytbl .name { width: 7em; text-align: left; }
#mytbl td.name { white-space: nowrap; overflow: hidden; text-overflow: ellipsis; cursor:pointer; }
#mytbl .eligiblewait { width: 4.7em; text-align: right; }
#mytbl .queuewait { width: 4.7em; text-align: right; }
#mytbl .runtime { width: 4.7em; text-align: right; }
#mytbl .cputime { width: 4.7em; text-align: right; }
#mytbl .actions { width: 2em; text-align: center; }
#mytbl .action { cursor:pointer; }

#mytbl td { line-height: 1.2em; }
#mytbl th { line-height: 24px; text-align: center !important; }

#mytbl th.restarts:hover,
#mytbl th.exitcode:hover,
#mytbl th.status:hover,
#mytbl th.step:hover,
#mytbl th.metric:hover,
#mytbl th.eligiblewait:hover,
#mytbl th.queuewait:hover
{ cursor: help; }

#mytbl a { text-decoration: default; color: default; }
</style>"""

STATE_NAMES2 = {"RUNNING": "🏃", "PENDING": "⏳", "REQUEUE_HOLD": "🚫", "COMPLETING": "🌇", "DONE": "✅", "FAILED": "❌", "UNKNOWN": "😕", "TIMEOUT": "⏰", "CANCELLED": "🛑"}

def fmt_sws_args(args):
    ss = []
    for a in args:
        lhs, rhs = a.split("=", 1)
        ss.append(f"{lhs}=<b>{rhs}</b>")
    return ' '.join(ss)

def hms(s, days=True):
    m, s = divmod(s, 60)
    h, m = divmod(m, 60)
    d, h = divmod(h, 24)
    if d:
        if days:
            return f"{d}-{h:02d}:{m:02d}:{s:02d}"
        else:
            return f"{d*24+h:02d}:{m:02d}:{s:02d}"
    elif h:
        return f"{h}h{m:02d}m{s:02d}s"
    elif m:
        return f"{m}m{s:02d}s"
    else:
        return f"{s}s"

def render_xid_view(configs, status, saccts, last_metrics, height=height, metric_to_show="train/loss"):
    thead = "<thead><tr><th class=wid>wid"
    thead += "<th class=jid>jid/logs"
    thead += "<th class=restarts title=Restarts>R"
    # thead += "<th class=exitcode title='Last run's exit code (0 = good)'>E"
    thead += f"<th class=status title='Job Status: {' '.join(f"{e}={t}" for t, e in STATE_NAMES2.items())}'>ST"
    thead += "<th class=step title='Last step in metrics'>Step"
    if metric_to_show:
        thead += f"<th class=metric title='Last value of {metric_to_show}'>M"
    thead += "<th class=conf>Config args"
    thead += "<th class=name>Name (wd)"
    # thead += "<th class=eligiblewait title='Time between job submission and becoming eligible for scheduling (init + dependencies)'>Ewait"
    thead += "<th class=queuewait title='Time between being eligible and getting scheduled (queueing for resources wait)'>Qwait"
    thead += "<th class=runtime>Runtime"
    thead += "<th class=actions title='Click to copy the command.'>Act"
    # thead += "<th class=cputime>CPU time"

    tbody = "<tbody>"
    for wid in sorted(configs):
        tbody += "<tr>"
        tbody += f"<td class=wid>{wid}"
        tbody += f"<td class=jid><a href='https://www.internalfb.com/fair_hub/job/FAIR_SC_3/{configs[wid]['jid']}/details' target=_blank>{configs[wid]['jid']}</a>"
        tbody += f"<td class=restarts>{saccts[wid]['restart_cnt']}"
        # tbody += f"<td class=exitcode>{saccts[wid]['exit_code']['return_code']['number']}"
        tbody += f"<td class=status title='{status[wid]} (Last exit code: {saccts[wid]['exit_code']['return_code']['number']})'>{STATE_NAMES2.get(status[wid], status[wid])}"
        tbody += f"<td class=step>{last_metrics[wid].get('step', 'n/a')}"
        if metric_to_show in last_metrics[wid]:
            tbody += f"<td class=metric>{last_metrics[wid][metric_to_show]:.2g}"
        else:
            tbody += "<td class=metric>n/a"
        tbody += f"<td class=conf>{fmt_sws_args(get_sws_args(saccts[wid]['submit_line']))}"
        tbody += f"<td class=name title='Click to copy the full name: {configs[wid]["name"]}' onclick='navigator.clipboard.writeText(this.textContent)'>{configs[wid]['name']}"
        # tbody += f"<td class=eligiblewait>{hms(saccts[wid]['time']['eligible'] - saccts[wid]['time']['submission'])}"
        if (tstart := saccts[wid]['time']['start']) == 4294967294:  # Means it got cancelled before it started.
            tbody += f"<td class=queuewait title='This job got cancelled before it got started'>never (?)"
        elif tstart == 0:
            tbody += f"<td class=queuewait title='This job got cancelled before it got started'>never (?)"
        else:
            tbody += f"<td class=queuewait>{hms(tstart - saccts[wid]['time']['eligible'])}"
        tbody += f"<td class=runtime>{hms(saccts[wid]['time']['elapsed'])}"
        # tbody += f"<td class=cputime>{hms(saccts[wid]['time']['total']['seconds'], days=False)}"
        tbody += f"<td class=actions>"
        if status[wid] in {"RUNNING", "PENDING", "REQUEUE_HOLD", "UNKNOWN"}:
            tbody += f"<span class=action title='scancel {configs[wid]["jid"]}' onclick='navigator.clipboard.writeText(this.title)'>⏹️</span>"
        else:
            tbody += f"<span class=action title='{Path(configs[wid].workdir).parent}/launch_{wid}.sh' onclick='navigator.clipboard.writeText(this.title)'>▶️</span>"
    return widgets.HTML(f"{css}<table id=mytbl width=100%>{thead}{tbody}</table>", layout={"margin": "0"})

# This would be directly rendering it:
# out = widgets.Output(layout={"height": f"{height}px", "overflow": "auto", "margin": "0"})
# out.append_display_data(render_xid_view(*get_xid_info("1029_160123"), height=512))
# display(out)

# Instead, we link up an editbox with the output and code-execution on edits.
# This is only because "hide cell code" has neater UX than the above alternative.
input_xid = widgets.Text(description='XID:', value=globals().get("XID"), placeholder='Enter xid, for example 1029_160123')
input_metric = widgets.Text(description='Metric:', value='train/loss')
refresh_button = widgets.Button(description='⟳', button_style='info', layout=widgets.Layout(width='unset'))
output_area = widgets.Output(layout={"height": f"{height}px", "overflow": "auto", "margin": "0"})

def on_change(change=None):
    xid = input_xid.value.strip(" '\"")
    metric = input_metric.value.strip(" '\"")
    with output_area:
        clear_output()
        print("Loading...", flush=True)
        html = render_xid_view(*get_xid_info(xid), height=512, metric_to_show=metric)
        clear_output()
        display(html)

input_xid.observe(on_change, names='value')
input_metric.observe(on_change, names='value')
refresh_button.on_click(lambda b: on_change())
display(widgets.VBox([widgets.HBox([input_xid, input_metric, refresh_button]), output_area]))

# Quick look at config and logs

### Code setup

In [330]:
from contextlib import contextmanager
from ipywidgets import Output
from IPython.display import display, display_html

# Workaround to fix weird scroll-inducing whitespace at end of many cell's outputs.
display_html("""<style>.jp-OutputArea { max-height: none !important; overflow-y: visible !important }</style>""", raw=True)

@contextmanager
def show_scrolling(height=200):
    out = Output(layout={"border": "1px solid #ccc", "height": f"{height}px", "overflow": "auto"})
    with out:
        yield
    display(out)

from pathlib import Path
import json
import sws

def print_config(run, height=384):
    workdir = Path('/checkpoint/rigi/bv2/workdirs') / run
    c = sws.Config(**json.loads((workdir / 'config.json').read_text())).finalize()
    with show_scrolling(height):
        print(c)


def print_logs(run, height=256, head=1000, tail=1000, linehead=100, linetail=100):
    def _snip_long_line(line, linehead=linehead, linetail=linetail):
        if len(line) > linehead + linetail:
            return line[:linehead] + " ... <SNIP> ... " + line[-linetail:]
        else:
            return line

    workdir = Path('/checkpoint/rigi/bv2/workdirs') / run
    c = sws.Config(**json.loads((workdir / 'config.json').read_text())).finalize()

    user = workdir.owner()
    full_log = Path(f'/checkpoint/rigi/bv2/slurm_out/{user}/{c.jid}.txt').read_text()
    loglines = full_log.split('\n')
    print(f"First {head} lines:")
    with show_scrolling(height):
        print('\n'.join(map(_snip_long_line, loglines[:head])))
    # import time
    # time.sleep(3)
    print(f"Last {tail} lines:")
    with show_scrolling(height):
        print('\n'.join(map(_snip_long_line, loglines[-tail:])))
    return full_log

### Actual look

In [329]:
print_config('1129_221439/fv-resize-1129_221439-0', height=384)

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

In [292]:
log = print_logs(f'{XID}/qkv-codewall-base-1117_163432-50', height=192, head=1500, tail=5000, linehead=200, linetail=200)

First 1500 lines:


Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

Last 5000 lines:


Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

# Tmp/dev

This is a place to dig deeper or figure out some things. The useful variables are: `jobs` (from `squeue` command), `hot_xids`, and `workdirs` (or sth like `basedir / workdirs[0]`).
Another useful function is `configs, states, saccts = get_xid_info("xid_string")`.

In [31]:
jobs.query('NAME == "1125_145646"')

,JOBID,NAME,USER,STATE,TIME,CPUS,QOS,NODES,TRES_PER_NODE,RESTART_COUNT,REASON
1,1113906,1125_145646,zhai,RUNNING,2-05:52:47,1536,h100_rigi_high,8,gres/gpu:8,1,None
2,1113905,1125_145646,zhai,RUNNING,2-05:53:50,1536,h100_rigi_high,8,gres/gpu:8,1,None


In [ ]:
configs, states, saccts = get_xid_info("1202_145629")

In [344]:
def pprint(x, prefix=""):
    if isinstance(x, (str, int, float)):
        print(f"{prefix}: {x}")
    elif isinstance(x, list):
        for i, y in enumerate(x):
            pprint(y, prefix=f"{prefix}.{i}")
    elif isinstance(x, dict):
        for k, y in x.items():
            pprint(y, prefix=f"{prefix}.{k}")
    else:
        print(f"??? {x}")

pprint(saccts[0])

.account: rigi
.comment.administrator: 
.comment.job: 
.comment.system: 
.allocation_nodes: 0
.array.job_id: 0
.array.limits.max.running.tasks: 0
.array.task_id.set: False
.array.task_id.infinite: False
.array.task_id.number: 0
.array.task: 
.association.account: rigi
.association.cluster: fair-sc-3
.association.partition: 
.association.user: zhai
.association.id: 571
.block: 
.cluster: fair-sc-3
.constraints: 
.container: 
.derived_exit_code.status.0: SUCCESS
.derived_exit_code.return_code.set: True
.derived_exit_code.return_code.infinite: False
.derived_exit_code.return_code.number: 0
.derived_exit_code.signal.id.set: False
.derived_exit_code.signal.id.infinite: False
.derived_exit_code.signal.id.number: 0
.derived_exit_code.signal.name: 
.time.elapsed: 0
.time.eligible: 1764858003
.time.end: 0
.time.planned.set: True
.time.planned.infinite: False
.time.planned.number: 66756
.time.start: 0
.time.submission: 1764857882
.time.suspended: 0
.time.system.seconds: 0
.time.system.microsecon